# 08. Conclusiones del Proyecto — Iteraciones 1 a 5
### Modelo Predictivo de Demanda CRUZBER — Resumen Ejecutivo para Negocio

---

Este documento recoge **todo lo que hemos construido, aprendido y concluido** a lo largo de las cinco primeras iteraciones del modelo predictivo de demanda de CRUZBER.

Está escrito para que cualquier persona del equipo, con o sin perfil técnico, pueda entender:
- Qué hemos hecho
- Por qué lo hemos hecho
- Qué ha funcionado y qué no
- Dónde estamos ahora
- Cuál es el camino más inteligente para seguir mejorando

---
## 1. El Problema que queremos resolver

CRUZBER necesita saber **cuántas unidades de cada producto se van a vender en cada municipio cada semana**.

Tener esa información con antelación permite:
- Planificar la producción y los pedidos a proveedores
- Evitar roturas de stock en los productos más demandados
- Reducir el exceso de inventario en productos de baja rotación
- Optimizar la distribución logística por zona geográfica

Sin un modelo, la planificación se basa en la experiencia del equipo comercial o en medias históricas simples. El objetivo es **sustituir esa intuición por una predicción cuantitativa y reproducible**.

### ¿Qué predice exactamente el modelo?

Para cada combinación de **semana + provincia + municipio + código de artículo**, el modelo predice cuántas unidades se venderán esa semana.

Por ejemplo: *¿Cuántas unidades del artículo X se venderán en Zaragoza durante la semana 20 de 2025?*

### ¿Con qué datos trabajamos?

| Fuente | Qué contiene | Periodo |
|---|---|---|
| Transacciones CRUZBER | Ventas históricas por artículo, municipio y semana | 2022–2024 |
| Clima (OpenMeteo) | Temperatura, precipitación, viento por semana | 2022–2024 |
| Calendario ciclista | Número y duración de pruebas ciclistas por semana | 2022–2024 |
| Maestro de artículos | Clasificación ABC, familia, subfamilia de producto | — |

El dataset final para modelado tiene **252.836 filas** (una por semana × municipio × artículo) y cubre los años 2022, 2023 y 2024.

---
## 2. Cómo medimos si el modelo es bueno

Antes de entrar en los resultados, es importante entender qué significan las métricas que usamos.

### MAE — Error Absoluto Medio
> *"De media, ¿cuántas unidades se equivoca el modelo?"*

Si el MAE es **0.65 unidades**, significa que en promedio el modelo se equivoca en 0.65 unidades por semana y artículo. Es la métrica más directa para negocio. Cuanto más bajo, mejor.

### MAPE — Error Porcentual Medio
> *"De media, ¿qué porcentaje del valor real representa el error?"*

Si el MAPE es **26%**, el modelo se equivoca de media en un 26% del valor real. Es útil para comparar entre artículos con volúmenes muy distintos. Un error de 1 unidad sobre una venta real de 2 es un 50% de error; ese mismo error sobre una venta de 100 es solo un 1%.

### R² — Coeficiente de Determinación
> *"¿Qué porcentaje de las variaciones en la demanda es capaz de explicar el modelo?"*

Va de 0 a 1. Un R² de **0.30** significa que el modelo explica el 30% de por qué la demanda sube o baja. El 70% restante responde a factores que el modelo no conoce (promociones, decisiones del equipo comercial, rupturas de stock, etc.).

### Overfitting — Cuando el modelo "memoriza" en lugar de "aprender"
> *"¿Funciona el modelo igual de bien con datos nuevos que con los que usó para aprender?"*

El overfitting ocurre cuando el modelo aprende demasiado bien los datos históricos pero luego falla al predecir periodos nuevos. Lo medimos comparando el error en el periodo de entrenamiento (2022-2023) con el error en el periodo de test (2024). Si la diferencia es grande, hay overfitting.

| Brecha RMSE train vs test | Diagnóstico |
|---|---|
| < 15% | Excelente generalización |
| 15% – 25% | Aceptable |
| 25% – 40% | Moderado — hay margen de mejora |
| > 40% | Alto — el modelo memoriza más de lo que aprende |

---
## 3. El Algoritmo: ¿Por qué CatBoost?

Existen decenas de algoritmos de machine learning. Probamos cuatro en la primera iteración y **CatBoost ganó** en todos los indicadores.

¿Por qué CatBoost es especialmente adecuado para los datos de CRUZBER?

| Característica de los datos | Por qué CatBoost lo maneja bien |
|---|---|
| Muchas variables categóricas (municipio, artículo, familia) | CatBoost las procesa nativamente sin necesidad de transformarlas manualmente |
| Alta cardinalidad (1.000+ artículos, 100+ municipios) | Usa un codificado interno que evita el ruido de métodos simples |
| Datos con outliers (picos de 400 unidades) | Es robusto frente a valores extremos gracias a sus árboles de decisión |
| Dataset relativamente pequeño (252K filas) | No necesita millones de datos para funcionar bien |

> CatBoost es como un analista de datos muy experimentado: no necesita que le expliques cada detalle, aprende solo las relaciones complejas entre variables.

---
## 4. Resumen de las 5 Iteraciones

Cada iteración parte de la anterior e introduce una mejora concreta. La filosofía es **cambiar una cosa cada vez** para saber exactamente qué funciona y qué no.

### Tabla comparativa global

| Iteración | Mejora introducida | MAE | MAPE | R² | Mejora MAE |
|---|---|---|---|---|---|
| **It1 — Baseline** | Modelo inicial con datos históricos básicos | 0.793 | — | 0.295 | — |
| **It2 — Rolling Mean** | Añadimos la tendencia reciente (media 4 semanas) | 0.773 | — | 0.330 | -2.5% |
| **It3 — Estacionalidad** | Añadimos las ventas de la misma semana del año anterior | 0.769 | — | 0.330 | -0.5% |
| **It4 — Log1p** | Transformamos matemáticamente la variable a predecir | 0.649 | 26.3% | 0.287 | **-15.6%** |
| **It5 — Optuna + Enc + Festivos** | Optimización automática + memoria histórica por SKU/zona | 0.641 | 26.0% | 0.288 | -1.2% |

**Mejora total acumulada: -19.2%** en el error medio (de 0.793 a 0.641 unidades por semana y artículo).

---
## 5. Iteración 1 — El punto de partida

### ¿Qué hicimos?

Construimos el primer modelo utilizando los datos históricos de ventas tal cual, sin ninguna transformación especial. Probamos cuatro algoritmos diferentes para elegir el mejor:

| Algoritmo | MAE | R² | Resultado |
|---|---|---|---|
| **CatBoost** | 0.793 | 0.295 | Ganador |
| Regresión Lineal | 0.840 | 0.268 | 2º |
| Random Forest | 0.813 | 0.208 | 3º |
| XGBoost | 0.800 | 0.149 | 4º |

### ¿Qué aprendimos?

CatBoost es claramente el mejor algoritmo para este problema. A partir de aquí, todas las iteraciones usan CatBoost como base.

El R² de 0.295 significa que el modelo explica el **29.5% de la varianza** de la demanda. Es un punto de partida razonable para un problema tan complejo como la predicción de demanda en retail.

### ¿Por qué no es suficiente?

El modelo en este punto solo conoce el histórico de ventas por artículo y municipio, el clima y los eventos ciclistas. No tiene memoria de lo que pasó la semana pasada ni del mismo periodo el año anterior. Es como si cada semana empezara de cero para el modelo.

---
## 6. Iteración 2 — Dándole memoria al modelo

### ¿Qué hicimos?

Añadimos una variable nueva: la **media de ventas de las últimas 4 semanas** para cada artículo en cada municipio.

### ¿Por qué?

La demanda no es aleatoria de semana en semana. Si un artículo lleva 4 semanas vendiéndose bien, probablemente seguirá vendiéndose bien la semana que viene. Esta información no estaba en el modelo anterior.

Es como darle al modelo la capacidad de mirar el retrovisor antes de hacer una predicción.

> **Ejemplo**: Si el artículo X en Zaragoza vendió 3, 4, 3 y 4 unidades en las últimas 4 semanas, la media móvil es 3.5. Eso le dice al modelo que la demanda base de ese artículo en esa plaza está en torno a 3-4 unidades.

### Resultado

El MAE bajó de 0.793 a **0.773** (-2.5%) y el R² subió de 0.295 a **0.330**. Pequeña pero consistente mejora.

**Detalle técnico importante**: la media se calcula con un desplazamiento de una semana (`shift(1)`). Esto garantiza que al predecir la semana N, el modelo solo usa información hasta la semana N-1. Si no hiciéramos esto, el modelo estaría haciendo trampa al ver el futuro.

---
## 7. Iteración 3 — Capturando la estacionalidad anual

### ¿Qué hicimos?

Añadimos otra variable nueva: las **ventas de la misma semana del año anterior** para cada artículo y municipio.

### ¿Por qué?

El deporte tiene una estacionalidad muy marcada. Las ventas de material de ciclismo o running en la semana 20 (finales de mayo) tienden a parecerse año tras año. La media móvil de las últimas 4 semanas captura la tendencia reciente, pero no recuerda lo que pasó hace un año.

> **Ejemplo**: Si el artículo X vendió 8 unidades en la semana 20 de 2023, es una señal relevante para predecir lo que venderá en la semana 20 de 2024.

### Resultado

El MAE apenas bajó (-0.5%, de 0.773 a **0.769**). La mejora fue marginal. ¿Por qué? Porque la cobertura de esta variable era limitada: solo teníamos datos del año anterior para una parte de los artículos y municipios. Para los que no tenían histórico del año anterior, el valor era cero, lo que añadía más ruido que señal.

Aun así, la variable quedó en el modelo porque **conceptualmente es correcta** y en iteraciones futuras, con más años de datos, será más valiosa.

---
## 8. Iteración 4 — El mayor salto: transformación matemática del objetivo

### ¿Qué hicimos?

Aplicamos una **transformación logarítmica** a la variable que queremos predecir (unidades).

### ¿Por qué fue necesario?

Imagina que tienes que predecir el peso de animales en un zoo. Si la mayoría son conejos (1-2 kg) pero también hay elefantes (5.000 kg), el modelo va a dedicar toda su capacidad a predecir bien los elefantes porque son los que tienen errores más grandes. El resultado: predice bien a los elefantes pero regular al resto.

Eso es exactamente lo que pasaba con los datos de CRUZBER:
- El **75% de las filas tienen 1 unidad** (la mayoría de pedidos son de 1 unidad)
- Pero hay pedidos de hasta **400 unidades** (grandes clientes B2B)

El modelo dedicaba demasiada energía a predecir esos picos extremos y era mediocre con el caso general.

### ¿Qué hace la transformación log1p?

La función `log1p(x) = log(1 + x)` comprime los valores grandes sin perder información:

| Valor original | Tras log1p | Efecto |
|---|---|---|
| 1 unidad | 0.69 | Casi igual |
| 10 unidades | 2.40 | Comprimido |
| 100 unidades | 4.61 | Muy comprimido |
| 400 unidades | 5.99 | Mínimo impacto |

Al predecir, el modelo trabaja en esta escala comprimida. Al final, revertimos la transformación con `expm1` para obtener las unidades reales.

### Resultado

El MAE bajó de 0.769 a **0.649** (**-15.6%**). Es la mejora más grande de todo el proyecto. Una sola transformación matemática logró más que las dos iteraciones anteriores juntas.

También introdujimos en esta iteración la **validación cruzada temporal** (TimeSeriesSplit con 3 periodos), que confirma que el modelo no depende del año concreto elegido para el test:

| Fold | MAE | MAPE | R² |
|---|---|---|---|
| Fold 1 | 0.656 | 29.2% | 0.410 |
| Fold 2 | 0.623 | 27.7% | 0.431 |
| Fold 3 | 0.657 | 25.9% | 0.312 |
| **Media** | **0.645 ± 0.019** | **27.6% ± 1.6%** | **0.384 ± 0.063** |

La desviación estándar del MAE es mínima (±0.019): el modelo es **muy estable** entre periodos.

---
## 9. Iteración 5 — Tres mejoras, resultados mixtos

### ¿Qué hicimos?

Introdujimos tres mejoras simultáneas:

1. **Optuna**: búsqueda automática de los mejores parámetros del modelo
2. **Target Encoding**: añadimos la demanda media histórica de cada artículo en cada municipio como variable explícita
3. **Festivos nacionales**: incorporamos el calendario de festivos oficiales de España

---

### Mejora 1 — Optuna: encontrar los mejores parámetros automáticamente

CatBoost tiene decenas de parámetros configurables. En las iteraciones anteriores usábamos valores estándar. Optuna es una herramienta que prueba automáticamente 30 combinaciones distintas de parámetros y encuentra la que minimiza el error.

Los parámetros más relevantes encontrados:

| Parámetro | Valor anterior | **Valor óptimo** | Qué significa |
|---|---|---|---|
| Velocidad de aprendizaje | 0.05 | **0.143** | El modelo aprende más rápido |
| Regularización L2 | 3 | **11.6** | Penaliza más la complejidad — reduce overfitting |
| Mínimo de datos por hoja | 1 | **74** | Cada decisión se toma con al menos 74 ejemplos |

> La regularización subió de 3 a 11.6 y el mínimo de datos por hoja de 1 a 74. Optuna nos estaba diciendo: *el modelo anterior era demasiado complejo para los datos disponibles*.

---

### Mejora 2 — Target Encoding: la memoria histórica por artículo y zona

Hasta ahora el modelo sabía que el artículo X existe y que Zaragoza es una ciudad, pero no tenía un número que dijera directamente *"este artículo en esta ciudad vende de media 3.2 unidades por semana"*.

El Target Encoding construye exactamente eso: una media histórica por cada combinación artículo-municipio, calculada de forma segura para no ver el futuro.

**Resultado**: esta variable llegó al **2º puesto de importancia con un 15.9%**, superando a la media móvil que llevaba siendo la más importante desde la Iteración 2. Junto con el target encoding de municipio (7.0%), acapara el **22.9% de la importancia total** del modelo.

---

### Mejora 3 — Festivos: resultado decepcionante

Incorporamos el calendario oficial de festivos nacionales de España esperando reducir los picos de error en semanas como Semana Santa o Navidad.

El resultado: importancia del **0.15%** — prácticamente irrelevante para el modelo.

**¿Por qué no funcionó?** Solo hay 23 semanas festivas en 3 años de datos. Es demasiado poco para que el modelo aprenda un patrón estadísticamente robusto. Para que los festivos funcionen, habría que incorporar también los festivos autonómicos por provincia (cada comunidad autónoma tiene los suyos propios), lo que cuadruplicaría la cobertura.

---

### Resultado global de Iteración 5

| Métrica | It4 | **It5** | Cambio |
|---|---|---|---|
| MAE | 0.649 | **0.641** | -1.2% |
| MAPE | 26.3% | **26.0%** | -0.3 pp |
| RMSE | 3.499 | **3.497** | ~0% |
| R² | 0.287 | **0.288** | ~0% |

Mejora real pero pequeña. El modelo ha llegado a un punto de **rendimientos decrecientes** con los datos actuales.

---
## 10. ¿Dónde estamos hoy? El diagnóstico global

### El modelo en cifras

```
Iteración 1  →  MAE 0.793  (punto de partida)
Iteración 2  →  MAE 0.773  (-2.5%)   ← memoria reciente (rolling mean)
Iteración 3  →  MAE 0.769  (-0.5%)   ← estacionalidad anual
Iteración 4  →  MAE 0.649  (-15.6%)  ← transformación logarítmica  ★ mayor salto
Iteración 5  →  MAE 0.641  (-1.2%)   ← Optuna + Target Encoding + Festivos
──────────────────────────────────────────────────────────────────────────────
Mejora total: -19.2% en MAE
```

El modelo se equivoca de media **0.64 unidades** por semana, artículo y municipio.

### Rendimiento por tipo de producto

No todos los productos son igual de predecibles. La clasificación ABC divide los artículos en:
- **Tipo A**: los productos estrella, alto volumen, los más importantes para la facturación
- **Tipo B y C**: la cola larga, muchos artículos con bajo volumen, comportamiento más errático

| Segmento | MAE | MAPE | R² | Interpretación |
|---|---|---|---|---|
| **Tipo A** | 0.684 uds | 27.3% | 0.320 | Predecible. El modelo explica 1 de cada 3 unidades de varianza |
| **Tipo B&C** | 0.543 uds | 23.1% | 0.211 | Volumen bajo, comportamiento errático. MAE bajo en absoluto pero R² pobre |

> Los productos Tipo B&C tienen un MAE en absoluto más bajo (0.543 vs 0.684) simplemente porque venden menos unidades. El error del 23% en MAPE indica que en proporción se equivoca tanto como en el Tipo A.

### Las semanas más difíciles de predecir (Tipo A)

El modelo tiene más dificultades en los cambios de temporada:

| Semana | MAE | Contexto probable |
|---|---|---|
| Semana 9 (feb-mar) | 1.245 uds | Cambio de invierno a primavera — inicio de temporada |
| Semana 19 (may) | 1.196 uds | Inicio temporada alta ciclista |
| Semana 36 (sep) | 1.127 uds | Vuelta al cole — reactivación post-verano |
| Semana 12 (mar) | 1.064 uds | Semana Santa — comportamiento atípico |

Estas semanas son exactamente donde **las promociones comerciales tienen más impacto**: cambios de temporada y periodos festivos suelen ir acompañados de acciones comerciales que el modelo no puede ver.

### El overfitting: diagnóstico honesto

| Iteración | Brecha RMSE train vs test | Diagnóstico |
|---|---|---|
| It3 | 50.7% | Alto |
| It4 | 38.8% | Moderado |
| **It5** | **47.8%** | Moderado — sin mejora |

El overfitting se ha reducido respecto a la Iteración 3 pero persiste. La conclusión más importante del proyecto hasta ahora:

> **El overfitting no es un problema técnico del modelo. Es una consecuencia de la falta de información.** El modelo aprende bien los patrones de 2022-2023, pero en 2024 hay eventos (promociones, cambios de precio, acciones comerciales) que no están en los datos y que no puede anticipar. Más regularización no resuelve esto: necesitamos datos nuevos.

---
## 11. Las variables más importantes del modelo

CatBoost nos dice cuánto contribuye cada variable a las predicciones. Esto es valioso porque nos revela **qué factores realmente mueven la demanda** según los datos.

| # | Variable | Importancia | Qué mide | Introducida en |
|---|---|---|---|---|
| 1 | Precio unitario | 16.0% | El precio al que se vendió ese artículo esa semana | It4 |
| 2 | **Target enc. SKU+Municipio** | **15.9%** | Demanda media histórica de cada artículo en cada municipio | **It5** |
| 3 | Media móvil 4 semanas | 12.8% | Tendencia de ventas de las últimas 4 semanas | It2 |
| 4 | Código de artículo | 8.0% | Identidad del producto | It1 |
| 5 | Municipio | 7.3% | Localización geográfica | It1 |
| 6 | **Target enc. Municipio** | **7.0%** | Demanda media histórica de cada municipio | **It5** |
| 7 | Canal de distribución | 5.7% | Distribución tradicional vs nuevos canales | It1 |
| 8 | Volatilidad 4 semanas | 4.8% | Cuánto varía la demanda semana a semana | It4 |
| 9 | Viento máximo | 1.9% | Condiciones meteorológicas | It1 |
| 10 | Temperatura media | 1.9% | Condiciones meteorológicas | It1 |

### Tres conclusiones de negocio

**1. El precio es el driver más importante (16.0%)**
El modelo ha aprendido que el precio explica más variación en la demanda que cualquier otra variable. Esto abre una pregunta estratégica: ¿estamos usando el precio como palanca activa en la gestión de la demanda?

**2. La memoria histórica por artículo+municipio es crítica (15.9%)**
La demanda base de cada artículo en cada municipio (cuánto vende habitualmente) es la segunda señal más valiosa. Esto confirma que la segmentación geográfica y por artículo es fundamental: no existe un patrón de demanda nacional uniforme.

**3. El clima tiene impacto limitado (1.9%)**
Temperatura y viento tienen poca importancia comparado con el histórico de ventas y el precio. Esto no quiere decir que el clima no importe — sin duda influye en si la gente practica deporte — sino que su efecto queda captado implícitamente a través de la estacionalidad histórica.

---
## 12. El hallazgo más importante del proyecto: los descuentos promocionales

Al analizar el dataset original de transacciones de CRUZBER, descubrimos una variable que no habíamos incorporado al modelo: **`por_descuento2`**, el descuento adicional o promocional aplicado sobre el precio base.

### ¿Qué es `por_descuento2`?

Las transacciones de CRUZBER tienen dos tipos de descuento:

| Variable | Qué es | Media | Correlación con ventas |
|---|---|---|---|
| `por_descuento` | Descuento comercial estándar de la tarifa | ~40% | 0.01 (irrelevante) |
| **`por_descuento2`** | **Descuento adicional / promocional** | 0% en el 98.7% de los casos | **0.206 (alto)** |

### El impacto de las promociones en la demanda

| Situación | Media de unidades vendidas | Multiplicador |
|---|---|---|
| Sin descuento promocional (`por_descuento2 = 0`) | 1.43 unidades | 1x (base) |
| **Con descuento promocional (`por_descuento2 > 0`)** | **6.76 unidades** | **4.7x** |

Cuando hay una promoción, la demanda se **multiplica por 4.7 de media**.

### ¿Por qué es tan importante para el modelo?

Recuerda las semanas más difíciles de predecir (semanas 9, 12, 19, 36). La hipótesis más probable es que esas semanas tienen un MAE alto precisamente porque hubo promociones o acciones comerciales que el modelo no podía ver.

Con una correlación de **0.206** con las unidades vendidas, `por_descuento2` es la señal individual más potente que hemos encontrado en todo el proyecto. Más que el clima, los eventos ciclistas y los festivos **juntos**.

> **En términos de negocio**: el modelo actual predice la demanda *orgánica* (sin intervención comercial). Incorporar los descuentos promocionales le permitirá predecir también la demanda *inducida* por acciones comerciales. Ese es el salto más grande que podemos dar.

---
## 13. Otra oportunidad identificada: Regiones Geográficas

### La idea

España tiene 52 provincias con climas radicalmente distintos. Agruparlas en regiones geográficas puede capturar patrones de comportamiento que van más allá de la provincia individual:

| Región | Provincias incluidas | Tipo de clima |
|---|---|---|
| **Noroeste** | Galicia, Asturias, Cantabria | Atlántico — lluvia frecuente, temperaturas suaves |
| **Norte** | País Vasco, Navarra, La Rioja, Aragón norte | Transición atlántico-continental |
| **Noreste** | Cataluña, Valencia, Baleares | Mediterráneo — veranos secos y calurosos |
| **Centro** | Madrid, Castilla y León, Castilla-La Mancha, Extremadura | Continental — inviernos fríos, veranos muy secos |
| **Sur** | Andalucía, Murcia | Mediterráneo-árido — la región más cálida |
| **Islas** | Canarias | Subtropical — demanda más estable todo el año |

### ¿Por qué puede ayudar al modelo?

Hay provincias con muy pocas observaciones en el dataset (Soria, Teruel, Zamora, Ávila). Para estas provincias, el modelo no tiene suficientes datos para aprender sus patrones individuales. Si las agrupamos en una región, el modelo puede aprender el patrón regional y aplicarlo a provincias con poco histórico.

Además, la interacción **región × temperatura** o **región × temporada alta** es más interpretable y estable que la temperatura puntual de un municipio concreto.

### ¿Cómo incorporarla?

La región se añadiría como una **variable adicional**, no en sustitución de la provincia. El modelo tendría así tres niveles de granularidad geográfica:
1. Región (macro) → captura patrones climáticos y culturales amplios
2. Provincia → captura diferencias dentro de la región
3. Municipio → captura el comportamiento local específico

---
## 14. Hoja de ruta — Próximos pasos

### Priorización por impacto

| Prioridad | Acción | Por qué | Impacto esperado | Esfuerzo |
|---|---|---|---|---|
| **1 — Crítica** | Incorporar `por_descuento2` al modelo | Correlación 0.206 con unidades, multiplica la demanda x4.7 | Muy alto | Bajo |
| **2 — Alta** | Regiones geográficas + interacción con clima | Mejora señal para provincias con poco histórico | Medio-alto | Bajo |
| **3 — Media** | Modelo específico para Tipo A | Segmento más importante, merece modelo propio | Medio | Medio |
| **4 — Media** | Festivos autonómicos por provincia | Los nacionales no funcionaron solos, ampliar cobertura | Medio | Bajo |
| **5 — Largo plazo** | Datos de stock disponible | Permite distinguir demanda potencial de demanda real | Alto | Alto |

### El mensaje más importante

El modelo ha mejorado un **19.2%** con ingeniería técnica sobre los datos existentes. Ese camino está llegando a su límite natural: las últimas dos iteraciones mejoran menos de un 2% cada una.

El siguiente salto significativo viene de **incorporar información de negocio que hoy no está en el modelo**: principalmente los descuentos promocionales (`por_descuento2`), que ya existen en los datos y tienen un impacto demostrado de 4.7x en la demanda.

Después, a medio plazo, incorporar el stock disponible permitiría pasar de predecir *demanda teórica* a predecir *demanda realizable*, que es el dato que realmente necesita la operación.

---

### Objetivo de las próximas iteraciones

```
Situación actual (It5):  MAE 0.641  |  R² 0.288  |  Brecha RMSE 47.8%

Objetivo It6 (descuentos + regiones):
    MAE < 0.58   |  R² > 0.38   |  Brecha RMSE < 35%

Objetivo It7 (modelo Tipo A + festivos autonómicos):
    MAE Tipo A < 0.60  |  R² Tipo A > 0.42
```

---
## 15. Conclusión Final

### Lo que hemos conseguido

En cinco iteraciones hemos construido un modelo de predicción de demanda que:

- **Reduce el error un 19.2%** respecto al punto de partida (de 0.793 a 0.641 unidades por semana y artículo)
- **Es estable**: funciona de forma consistente en distintos periodos de tiempo (variación de ±0.019 en MAE entre folds de validación cruzada)
- **Identifica los drivers clave**: precio, demanda histórica por artículo/municipio y tendencia reciente son los tres factores que más explican la demanda
- **Ha revelado el driver oculto**: las promociones (`por_descuento2`) multiplican la demanda por 4.7x y no están en el modelo — eso es el próximo paso más importante

### Lo que el modelo sabe hacer bien

- Predecir la demanda *orgánica* de artículos con histórico estable
- Capturar la estacionalidad anual y la tendencia reciente
- Distinguir el comportamiento por municipio y canal de distribución
- Comportarse de forma más precisa en los productos Tipo A (los más importantes)

### Lo que el modelo todavía no sabe hacer

- Anticipar el efecto de una promoción o acción comercial
- Predecir bien los picos de demanda extremos (semanas de cambio de temporada con acciones comerciales)
- Distinguir si una venta baja fue por falta de demanda o por rotura de stock

### El camino que queda

El modelo está preparado para dar el siguiente salto. Los datos necesarios existen (los descuentos promocionales están en el dataset original de CRUZBER). El trabajo técnico para incorporarlos es relativamente sencillo. El impacto potencial es el mayor de todo el proyecto.

> *Un modelo de demanda que no sabe que hay una promoción activa es como un meteorólogo que predice el tiempo sin conocer la presión atmosférica: puede acertar en los días normales, pero fallará siempre en las tormentas.*